# Dependencies install (for topo)

Restart session after installing

In [ ]:
%%capture
!sudo apt-get install graphviz graphviz-dev

In [ ]:
%%capture
%pip install kmapper networkx graphviz pygraphviz umap-learn giotto-tda
%pip install scikit-learn==1.6.1 numpy==1.26.4

# Imports

In [ ]:
from IPython import display
import logging
import sys

BASE_FORMAT = '[%(asctime)s]%(levelname)s: %(message)s'
BASE_DATE_FORMAT = '%Y-%m-%d %H:%M:%S'

logger = logging.getLogger()
logger.handlers = []
handler = logging.StreamHandler(sys.stdout)
handler.setFormatter(logging.Formatter(fmt=BASE_FORMAT, datefmt=BASE_DATE_FORMAT))
handler.setLevel(logging.INFO)
logger.addHandler(handler)
logger.setLevel(logging.INFO)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
import tensorflow_probability as tfp
import sklearn
from sklearn.preprocessing import MinMaxScaler
from tqdm import tqdm
import logging
import time
import json

plt.ioff()


logger.info("numpy = " + str(np.__version__))
logger.info("tf = " + str(tf.__version__))
logger.info("sklearn = " + str(sklearn.__version__))
logger.info("GPUs: = " + str(tf.config.list_physical_devices('GPU')))

[2025-05-14 07:15:58]INFO: numpy = 1.26.4
[2025-05-14 07:15:58]INFO: tf =2.18.0
[2025-05-14 07:15:58]INFO: sklearn =1.6.1
[2025-05-14 07:15:58]INFO: GPUs: = []


# Mapper Graph

In [ ]:
import networkx as nx
from collections import deque
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE, Isomap
from sklearn.cluster import DBSCAN, OPTICS, HDBSCAN
from scipy.spatial.distance import pdist
from scipy.sparse.csgraph import floyd_warshall
from sklearn.metrics import pairwise_distances
from tqdm import tqdm

# Giotto-tda
from gtda.mapper import ParallelClustering, CubicalCover, Nerve
from gtda.mapper.utils.pipeline import identity


class KMProcess:
    def __init__(self, verbose=1, n_jobs=-1):
        self.verbose = verbose
        self.n_jobs = n_jobs
        self.name = ''
        self.used_project_method = ''
        self.used_project_params = ''
        self.used_cluster_method = ''
        self.data_shape = None
        self.projection = None
        self.projected_data = None
        self.graph = None
        self.components = []
        # Data for training
        self.num_components = None
        self.data2cluster_idx = None
        self.cluster2component_idx = None
        self.cluster_component2matrix_idx = None
        self.dist_matrices = None
        self.components_sigmas = None

    def project(self, data, method: str, **kwargs):
        proj = None
        method_params = kwargs.copy()
        if "n_jobs" not in method_params:
            method_params["n_jobs"] = self.n_jobs
        if "verbose" not in method_params:
            method_params["verbose"] = self.verbose

        if method.lower() == 'isomap':
            method_params.pop("verbose")
            self.projection = Isomap(**method_params)
        elif method.lower() == 'tsne':
            self.projection = TSNE(**method_params)
        elif method.lower() == 'umap':
            import umap
            self.projection = umap.UMAP(**method_params)
        elif method.lower() == 'pca':
            self.projection = PCA(**kwargs)
        else:
            self.projection = method

        self.data_shape = data.shape
        self.used_project_method = str(method)
        self.used_project_params = "_".join(
            [f"{k}_{v}" for k, v in kwargs.items()]
        ).replace('.', '_')

        logger.info(f"Projecting using {self.used_project_method} with params {method_params}")

        self.projected_data = self.projection.fit_transform(data)
        return self.projected_data

    def plot_projections(self, dot_size=0.5, colors=None, with_legend=True, cols=(0,1)):
        if self.projected_data is None:
            return

        plt.figure(figsize=(8 + int(colors is not None)*2, 8))

        if colors is None or not with_legend:
            sc = plt.scatter(
                self.projected_data[:, cols[0]],
                self.projected_data[:, cols[1]],
                s=dot_size,
                c=colors
            )
            if colors is not None:
                plt.colorbar(sc)
        else:
            for i in np.unique(colors):
                inds = colors == i
                plt.scatter(
                    self.projected_data[inds][:, cols[0]],
                    self.projected_data[inds][:, cols[1]],
                    s=dot_size,
                    label=str(i)
                )
            lgnd = plt.legend()
            for i in range(len(lgnd.legend_handles)):
                lgnd.legend_handles[i]._sizes = [30]

        plt.title(f"Projected {self.used_project_method} {self.name}")
        plt.savefig(f"{self.name}_projected_data_{self.used_project_method}_{self.used_project_params}.png")
        plt.show()


    def map_data(self, data=None, cluster_method='DBSCAN', cover_n=10, cover_p=0.2, min_intersection=1, **kwargs):
        # Density based clusering probably will be the best for this task
        if cluster_method.lower() == "dbscan":
            clusterer = DBSCAN(**kwargs)
        elif cluster_method.lower() == "hdbscan":
            clusterer = HDBSCAN(**kwargs)
        elif cluster_method.lower() == "optics":
            clusterer = OPTICS(**kwargs)
        else:
            clusterer = cluster_method

        self.used_cluster_method = str(cluster_method)

        logger.info(f"Preforming cubical covering with {cover_n} intervals and {cover_p*100}% overlap")

        cover = CubicalCover(n_intervals=cover_n, overlap_frac=cover_p)
        cover_masks = cover.fit_transform(self.projected_data)

        logger.info(f"Covered with {cover_masks.shape[1]} non-empty hypercubes")
        logger.info(f"Performing clustering with {self.used_cluster_method}")

        if data is None:
            data = self.projected_data
            logger.warning("Clustering on projected data")

        p_cluster = ParallelClustering(clusterer, n_jobs=self.n_jobs)
        clusters = p_cluster.fit_transform([data, cover_masks])

        logger.info(f"Clustring complete")
        logger.info(f"Create graph")

        nerve = Nerve(min_intersection=min_intersection)
        graph = nerve.fit_transform(clusters)

        self.graph = {
            "nodes": list(graph.vs['node_elements']),
            "edges": list(map(lambda x: x.tuple, graph.es)),
        }
        params_str = f"{cluster_method}_{cover_n}_{cover_p}".replace('.', '_')
        filename = f"{self.name}_KMapper_Graph_{params_str}.json"
        self.save_graph(filename)

        return graph

    def save_graph(self, filename):
        self.graph['nodes'] = list(map(lambda x: x.tolist(), self.graph['nodes']))
        with open(filename, "w") as f:
            json.dump(self.graph, f)
            logger.info(f"Saved graph to {filename}")
        self.graph['nodes'] = list(map(np.array, self.graph['nodes']))

    def load_graph(self, graph_path):
        with open(graph_path, "r") as f:
            self.graph = json.load(f)
        self.graph['nodes'] = list(map(np.array, self.graph['nodes']))
        return self.graph

    def _get_nx_graph_with_positions(self):
        G = nx.Graph()
        G.add_nodes_from(np.arange(len(self.graph['nodes'])))
        G.add_edges_from(self.graph['edges'])
        pos = nx.nx_agraph.graphviz_layout(G)
        return G, pos

    def plot_graph(self, colors=None):
        G, positions = self._get_nx_graph_with_positions()

        if not nx.is_connected(G):
            logger.warning("Graph is not connected!")

        if colors is not None:
            node_colors = []
            for inds in self.graph['nodes']:
                cluter_colors = colors[inds].astype(np.int32)
                most_freq = np.argmax(np.bincount(cluter_colors))
                node_colors.append(most_freq)
        else:
            node_colors = 'lightblue'

        sizes = np.array([len(inds) for inds in self.graph['nodes']])
        sizes[sizes < 30] = 30 # set min node size
        sizes[sizes > 120] = 120 # set max node size

        # Draw using precomputed positions
        nx.draw(
            G,
            positions,
            with_labels=False,
            node_color=node_colors,
            node_size=sizes,
            edge_color='gray',
            linewidths=0.1,
            font_size=10
        )
        plt.title(f"Cluster Graph of {self.name}")
        plt.savefig(f"{self.name}_graph_visu.png")
        plt.show()

    def _apd(self, graph_matrix):
        """Compute the shortest-paths lengths."""
        A = graph_matrix
        n = graph_matrix.shape[0]
        if all(A[i][j] for i in range(n) for j in range(n) if i != j):
            return A
        Z = A @ A
        B = np.array([
            [1 if i != j and (A[i][j] == 1 or Z[i][j] > 0) else 0 for j in range(n)]
            for i in range(n)
        ])
        T = self._apd(B)
        X = T @ A
        degree = [sum(A[i][j] for j in range(n)) for i in range(n)]
        D = np.array([
            [
                2 * T[i][j] if X[i][j] >= T[i][j] * degree[j] else 2 * T[i][j] - 1
                for j in range(n)
            ]
            for i in range(n)
        ])
        return D


    def process_components_bfs(self, matrix):
        n = matrix.shape[0]
        visited = np.zeros(n, dtype=bool)
        self.cluster2component_idx = np.zeros(n, dtype=np.uint16)
        self.components = []
        for i in range(n):
            if not visited[i]:
                queue = deque([i])
                visited[i] = True
                component = [i]
                while queue:
                    node = queue.popleft()
                    neighbors = np.where(matrix[node] > 0)[0]
                    for neighbor in neighbors:
                        if not visited[neighbor]:
                            visited[neighbor] = True
                            component.append(neighbor)
                            queue.append(neighbor)
                component = np.array(component)
                self.cluster2component_idx[component] = len(self.components)
                self.components.append(component)

        self.num_components = len(self.components)

        self.cluster_component2matrix_idx = np.zeros(n, dtype=np.uint16)
        for comp_ids in self.components:
            self.cluster_component2matrix_idx[comp_ids] = np.arange(comp_ids.shape[0])

        return self.components

    def connect_outliers(self, connect_outliers_threshold: int = 0):
        if connect_outliers_threshold <= 0:
            return []

        added_edges = []
        deleted_components = []

        for comp_ind in range(len(self.components)):
            comp = self.components[comp_ind]
            if len(comp) > 1 or len(self.graph['nodes'][comp[0]]) > connect_outliers_threshold:
                continue
            # Get cluster projections
            cluster_ind = comp[0]
            data_inds = self.graph['nodes'][comp[0]]
            cluster_projections = self.projected_data[data_inds]
            # Take average as point
            avg_cluster_projection = cluster_projections.mean(axis=0)[None, :]
            # Take all other projecion points
            other_projections = np.delete(self.projected_data, data_inds, axis=0)
            # Get dists from avg cluster point to all others
            cluster_dists = pairwise_distances(avg_cluster_projection, other_projections)[0]
            # Find nearest cluster
            nearest_data_ind = np.argmin(cluster_dists)
            nearest_cluster_ind = self.data2cluster_idx[nearest_data_ind]
            nearest_cluster_comp_ind = self.cluster2component_idx[nearest_cluster_ind]
            # Connect cluster to nearest cluster component
            new_edge = (int(nearest_cluster_ind), int(cluster_ind))
            self.graph['edges'].append(new_edge)
            added_edges.append(new_edge)
            # Add to nearest cluster component
            self.components[nearest_cluster_comp_ind] = np.concatenate(
                [self.components[nearest_cluster_comp_ind], [cluster_ind]]
            )
            self.cluster2component_idx[cluster_ind] = nearest_cluster_comp_ind
            deleted_components.append(comp_ind)

        # Delete and renumerate components
        if deleted_components:
            self.components = [
                comp for comp_ind, comp in enumerate(self.components)
                if comp_ind not in deleted_components
            ]
            for comp_ind, comp in enumerate(self.components):
                self.cluster2component_idx[comp] = comp_ind
                self.cluster_component2matrix_idx[comp] = np.arange(comp.shape[0])
            self.num_components = len(self.components)

        self.save_graph(f"{self.name}_KMapper_Graph_outliers_connected.json")

        return added_edges


    def create_data_cluster_map(self):
        if self.data_shape is not None:
            self.data2cluster_idx = np.zeros(
                self.data_shape[0], dtype=np.uint32
            )
        else:
            self.data2cluster_idx = np.zeros(
                max(map(max, self.graph['nodes'])) + 1, dtype=np.uint32
            )
        nodes_cluster = [[] for _ in range(self.data2cluster_idx.shape[0])]

        for cluster_id, nodes_ids in enumerate(self.graph['nodes']):
            for node_id in nodes_ids:
                nodes_cluster[node_id].append(cluster_id)

        for node_id, node_clusters in enumerate(nodes_cluster):
            if len(node_clusters) == 1:
                self.data2cluster_idx[node_id] = node_clusters[0]
                continue
            # Add to cluster with probability inversed by cluser size
            cluster_sizes = np.array(
                [len(self.graph['nodes'][cl_ind]) for cl_ind in node_clusters],
                dtype=np.float32
            )
            cluster_inverse = 1 / cluster_sizes
            cluster_prob = cluster_inverse / cluster_inverse.sum()
            cluster_prob = np.cumsum(cluster_prob)
            cluster_prob[-1] = 1 # adjust in case float sum error
            cluster_ind = np.where(np.random.rand() < cluster_prob)[0][0]

            self.data2cluster_idx[node_id] = node_clusters[cluster_ind]

        return self.data2cluster_idx

    def _get_edge_dist(self, data_or_dist_matrix, source, dest):
        source_nodes = self.graph['nodes'][source]
        dest_nodes = self.graph['nodes'][dest]
        n = self.data_shape[0]

        if len(data_or_dist_matrix.shape) == 2:
            dists = pairwise_distances(
                data_or_dist_matrix[source_nodes],
                data_or_dist_matrix[dest_nodes]
            )
        else:
            sources, dests = np.meshgrid(source_nodes, dest_nodes)
            sources, dests = sources.flatten(), dests.flatten()
            dists = np.zeros(len(source_nodes) * len(dest_nodes), dtype=np.float32)
            for i in range(len(sources)):
                if sources[i] == dests[i]:
                    continue
                s, d = sources[i], dests[i] if sources[i] < dests[i] else dests[i], sources[i]
                ind = (n * s + d - ((s + 2) * (s + 1)) // 2)
                dists[i] = data_or_dist_matrix[ind]
            dists = dists.reshape((len(source_nodes), len(dest_nodes)))

        # Formula by https://research.math.osu.edu/tgda/mapperPBG.pdf
        source_avg = np.min(dists, axis=0).mean()
        dest_avg = np.min(dists, axis=1).mean()
        return max(source_avg, dest_avg)


    def get_dist_matrices(self, data=None, scale_param: float = -1, connect_outliers_threshold: int = 0):
        if self.data2cluster_idx is None:
            self.create_data_cluster_map()

        num_nodes = len(self.graph['nodes'])
        if data is None:
            graph_matrix = np.zeros((num_nodes, num_nodes), dtype=np.int32)
            for source, dest in self.graph['edges']:
                graph_matrix[source][dest] = 1
                graph_matrix[dest][source] = 1
        else:
            self.data_shape = data.shape
            graph_matrix = np.zeros((num_nodes, num_nodes), dtype=np.float32)

            logger.info("Calculating edges weights by data...")

            for source, dest in tqdm(self.graph['edges']):
                edge_dist = self._get_edge_dist(data, source, dest)
                graph_matrix[source][dest] = edge_dist
                graph_matrix[dest][source] = edge_dist

        logger.info("Prepared graph matrix with edges weights " + ("= 1" if data is None else "by data"))

        self.process_components_bfs(graph_matrix)

        logger.info(f"Found {self.num_components} components")

        if connect_outliers_threshold > 0:
            logger.info("Connect outliers...")
            added_edges = self.connect_outliers(connect_outliers_threshold)
            # Add edges to graph matrix
            for source, dest in added_edges:
                edge_weight = self._get_edge_dist(data, source, dest) if data is not None else 1
                graph_matrix[source][dest] = edge_weight
                graph_matrix[dest][source] = edge_weight
            logger.info(f"Outliers connected, new total number of components is {self.num_components}")

        self.dist_matrices = []
        dist_maxes = []
        for comp_ids in self.components:
            if data is None:
                comp_dist_matrix = self._apd(graph_matrix[comp_ids][:, comp_ids])
            else:
                comp_dist_matrix = floyd_warshall(
                    # Make C_CONTIGUOUS, required by function
                    graph_matrix[comp_ids][:, comp_ids].copy(order='C')
                )
            dist_maxes.append(comp_dist_matrix.max())
            self.dist_matrices.append(comp_dist_matrix)

        logger.info(f"Max dists in each component in graph: {dist_maxes}")

        self.dist_matrices = self.rescale_dist_matrices(scale_param, dist_maxes)

        return self.dist_matrices

    def rescale_dist_matrices(self, scale_param: float, dist_maxes=None):
        if scale_param <= 0:
            return self.dist_matrices

        new_dist_maxes = []
        for i, dist_matrix in enumerate(self.dist_matrices):
            dist_max = dist_maxes[i] if dist_maxes is not None else dist_matrix.max()
            if dist_max > scale_param:
                self.dist_matrices[i] = dist_matrix / dist_max * scale_param
                dist_max = scale_param
            new_dist_maxes.append(dist_max)

        logger.info(f"Scaled dist matrices of each component to max: {new_dist_maxes}")
        self.calc_components_sigmas()

        return self.dist_matrices

    def calc_components_sigmas(self):
        self.components_sigmas = np.ones(len(self.components), dtype=np.float32)
        for comp_ind, dist_matrix in enumerate(self.dist_matrices):
            if dist_matrix.shape[0] > 1:
                center_ind = np.argmin(np.max(dist_matrix, axis=1))
                center_dists = np.delete(dist_matrix[center_ind], center_ind)
                sigma = np.square(center_dists).sum() / len(center_dists)
                self.components_sigmas[comp_ind] = sigma

        logger.info(f"Sigmas squares of each component: {self.components_sigmas}")
        return self.components_sigmas

    def pipeline(self, data, config_file, colors=None):
        logger.info("Loading config file...")
        with open(config_file, "r") as f:
            config = json.load(f)

        self.name = config["name"]
        km_config = config["km"]
        plot_config = config["plot"]

        logger.info("Project data...")
        self.project(data, method=km_config["project_method"], **km_config.get("project_params", {}))
        if plot_config.get("plot_project", False):
            self.plot_projections(colors=colors, **plot_config.get("plot_project_params", {}))

        logger.info("Map data...")
        if km_config.get("map_use_data", True):
            self.map_data(data=data, **km_config["map_params"])
        else:
            self.map_data(**km_config["map_params"])

        if plot_config.get("plot_graph", False):
            self.plot_graph(colors=colors)

        logger.info("Create data to cluster map...")
        self.create_data_cluster_map()
        logger.info("Calc dist matrix...")
        scale_param = km_config.get("dist_matrix_scale", -1)
        connect_outliers_threshold = km_config.get("connect_outliers_threshold", 0)
        if km_config.get("dists_by_data", True):
            dist_matrices = self.get_dist_matrices(
                data, scale_param=scale_param, connect_outliers_threshold=connect_outliers_threshold
            )
        else:
            dist_matrices = self.get_dist_matrices(
                scale_param=scale_param, connect_outliers_threshold=connect_outliers_threshold
            )

        logger.info(f"Dist matrices shape: {[dm.shape for dm in dist_matrices]}")
        logger.info("Done!")

# VAE Models

In [ ]:
class BaseVAE(tf.keras.Model):
    """ Simple Variational autoencoder."""
    def __init__(self, input_size, latent_dim, sigmoid_applied=True):
        super(BaseVAE, self).__init__()
        self.latent_dim = latent_dim
        self.input_size = input_size
        self.sigmoid_applied = sigmoid_applied
        self.encoder = None
        self.decoder = None
        self.is_conditional = False

    @tf.function
    def sample(self, eps=None):
        if eps is None:
            eps = tf.random.normal(shape=(100, self.latent_dim))
        return self.decode(eps, apply_sigmoid=not self.sigmoid_applied)

    def encode(self, x):
        mean, logvar = tf.split(self.encoder(x), num_or_size_splits=2, axis=1)
        return mean, logvar

    def reparameterize(self, mean, logvar):
        eps = tf.random.normal(shape=mean.shape)
        return eps * tf.exp(logvar * .5) + mean

    def decode(self, z, apply_sigmoid=False):
        logits = self.decoder(z)
        if apply_sigmoid:
            probs = tf.sigmoid(logits)
            return probs
        return logits


class VAE(BaseVAE):
    """ Simple Variational autoencoder."""
    def __init__(self, input_size, latent_dim, hidden_layers=[16, 16]):
        super(VAE, self).__init__(input_size, latent_dim)
        self.encoder = tf.keras.Sequential(
            [
                tf.keras.layers.InputLayer(input_shape=(input_size,)),
                *[
                    tf.keras.layers.Dense(units, activation=tf.nn.silu)
                    for units in hidden_layers
                ],
                # No activation
                tf.keras.layers.Dense(latent_dim + latent_dim),
            ]
        )
        self.decoder = tf.keras.Sequential(
            [
                tf.keras.layers.InputLayer(input_shape=(latent_dim,)),
                *[
                    tf.keras.layers.Dense(units, activation=tf.nn.silu)
                    for units in hidden_layers[::-1]
                ],
                tf.keras.layers.Dense(input_size)
            ]
        )


class ConvVAE(BaseVAE):
    """Convolutional Variational Autoencoder."""

    def __init__(
            self, input_size, latent_dim,
            hidden_filters=[32, 64], hidden_layers=[],
            apply_sigmoid=False
    ):
        super(ConvVAE, self).__init__(input_size, latent_dim, apply_sigmoid)

        self.encoder = tf.keras.Sequential([
            tf.keras.layers.InputLayer(input_shape=input_size),
            *[
                tf.keras.layers.Conv2D(
                    filters=filters, kernel_size=3, strides=(2, 2),
                    padding="same", activation='relu'
                ) for filters in hidden_filters
            ],
            tf.keras.layers.Flatten(),
            *[
                tf.keras.layers.Dense(units, activation=tf.nn.silu)
                for units in hidden_layers
            ],
            # No activation
            tf.keras.layers.Dense(latent_dim + latent_dim),
        ])

        n_filt = len(hidden_filters)
        dec_units = input_size[0] * input_size[1] * hidden_filters[-1] // 2**(2 * n_filt + 1)
        dec_units_shape = (input_size[0] // 2**n_filt, input_size[1] // 2**n_filt, hidden_filters[-1] // 2)

        self.decoder = tf.keras.Sequential([
            tf.keras.layers.InputLayer(input_shape=(latent_dim,)),
            *[
                tf.keras.layers.Dense(units, activation=tf.nn.silu)
                for units in hidden_layers[::-1]
            ],
            tf.keras.layers.Dense(units=dec_units, activation=tf.nn.relu),
            tf.keras.layers.Reshape(target_shape=dec_units_shape),
            *[
                tf.keras.layers.Conv2DTranspose(
                    filters=filters, kernel_size=3, strides=(2, 2),
                    padding='same', activation='relu'
                ) for filters in hidden_filters[::-1]
            ],
            # No activation
            tf.keras.layers.Conv2DTranspose(
                filters=input_size[-1], kernel_size=3, strides=1, padding='same',
                activation="sigmoid" if apply_sigmoid else None
            )
        ])

In [ ]:
class BaseCVAE(BaseVAE):
    """ Simple Conditional Variational autoencoder."""
    def __init__(self, input_size, latent_dim, num_components, sigmoid_applied=True):
        super(BaseCVAE, self).__init__(input_size, latent_dim, sigmoid_applied)
        self.num_components = num_components
        self.data_encoder = None
        self.is_conditional = True

    def one_hot(self, y):
        return tf.one_hot(y, depth=self.num_components)

    def encode(self, x, y):
        encoded_data = self.data_encoder(x) if self.data_encoder is not None else x
        encoded_data = tf.concat([encoded_data, self.one_hot(y)], axis=-1)
        return super().encode(encoded_data)

    def decode(self, z, y, apply_sigmoid=False):
        z = tf.concat([z, self.one_hot(y)], axis=-1)
        return super().decode(z, apply_sigmoid)

    def sample(self, y, eps=None):
        if eps is None: # TODO sample with given by component sigma
            eps = tf.random.normal(shape=(100, self.latent_dim))
        return self.decode(eps, y, apply_sigmoid=not self.sigmoid_applied)


class CVAE(BaseCVAE):
    """ Conditional Variational autoencoder."""
    def __init__(self, input_size, latent_dim, num_components, hidden_layers=[16, 16]):
        super(CVAE, self).__init__(input_size, latent_dim, num_components)
        self.encoder = tf.keras.Sequential(
            [
                tf.keras.layers.InputLayer(input_shape=(input_size + num_components,)),
                *[
                    tf.keras.layers.Dense(units, activation=tf.nn.silu)
                    for units in hidden_layers
                ],
                # No activation
                tf.keras.layers.Dense(latent_dim + latent_dim),
            ]
        )
        self.decoder = tf.keras.Sequential(
            [
                tf.keras.layers.InputLayer(input_shape=(latent_dim + num_components,)),
                *[
                    tf.keras.layers.Dense(units, activation=tf.nn.silu)
                    for units in hidden_layers[::-1]
                ],
                tf.keras.layers.Dense(input_size)
            ]
        )


class ConvCVAE(BaseCVAE):
    """Conditional Convolutional Variational Autoencoder."""

    def __init__(
            self, input_size, latent_dim, num_components,
            hidden_filters=[32, 64], hidden_layers=[64],
            apply_sigmoid=False
    ):
        super(ConvCVAE, self).__init__(input_size, latent_dim, num_components, apply_sigmoid)

        self.data_encoder = tf.keras.Sequential([
            tf.keras.layers.InputLayer(input_shape=input_size),
            *[
                tf.keras.layers.Conv2D(
                    filters=filters, kernel_size=3, strides=(2, 2),
                    padding="same", activation='relu'
                ) for filters in hidden_filters
            ],
            tf.keras.layers.Flatten()
        ])

        n_filt = len(hidden_filters)
        enc_out_units = input_size[0] * input_size[1] * hidden_filters[-1] // 2**(2 * n_filt)

        self.encoder = tf.keras.Sequential([
            tf.keras.layers.InputLayer(input_shape=(enc_out_units + num_components,)),
            *[
                tf.keras.layers.Dense(units, activation=tf.nn.silu)
                for units in hidden_layers
            ],
            # No activation
            tf.keras.layers.Dense(latent_dim + latent_dim),
        ])

        dec_units = enc_out_units // 2
        dec_units_shape = (input_size[0] // 2**n_filt, input_size[1] // 2**n_filt, hidden_filters[-1] // 2)

        self.decoder = tf.keras.Sequential([
            tf.keras.layers.InputLayer(input_shape=(latent_dim + num_components,)),
             *[
                tf.keras.layers.Dense(units, activation=tf.nn.silu)
                for units in hidden_layers[::-1]
            ],
            tf.keras.layers.Dense(units=dec_units, activation=tf.nn.relu),
            tf.keras.layers.Reshape(target_shape=dec_units_shape),
            *[
                tf.keras.layers.Conv2DTranspose(
                    filters=filters, kernel_size=3, strides=(2, 2),
                    padding='same', activation='relu'
                ) for filters in hidden_filters[::-1]
            ],
            tf.keras.layers.Conv2DTranspose(
                filters=input_size[-1], kernel_size=3, strides=1, padding='same',
                activation="sigmoid" if apply_sigmoid else None
            )
        ])

In [ ]:
def get_model_from_config(config_path, kmprocess=None):
    with open(config_path, "r") as f:
        config = json.load(f)["model"]

    is_conditional = config.get("is_conditional", False)
    if is_conditional and kmprocess is None:
        raise Exception("Provide KMprocss for building conditional vae")
    elif is_conditional:
        num_components = kmprocess.num_components
        if num_components == 1:
            logger.warning("Number of components is 1, CVAE replaced with VAE")
            is_conditional = False
    else:
        num_components = 0

    if config["type"].lower() == "vae":
        params = {
            "input_size": int(config["input_size"]),
            "latent_dim": config["latent_dim"],
            "hidden_layers": config["hidden_layers"]
        }
        if is_conditional:
            model = CVAE(num_components=num_components, **params)
        else:
            model = VAE(**params)
    elif config["type"].lower() in ("convvae", "conv_vae"):
        params = {
            "input_size": tuple(config["input_size"]),
            "latent_dim": config["latent_dim"],
            "hidden_filters": config["hidden_filters"],
            "hidden_layers": config.get("hidden_layers", []),
            "apply_sigmoid": config.get("apply_sigmoid", False)
        }
        if is_conditional:
            model = ConvCVAE(num_components=num_components, **params)
        else:
            model = ConvVAE(**params)
    else:
        logger.error(f"Not supported type: {config['type']}")
        return None

    return model

# Train Pipeline

In [ ]:
class TrainPipeline:
    def __init__(self, model, config_path, kmprocess=None, verbose=1):
        self.model = model
        self.config_path = config_path
        self.verbose = verbose

        with open(config_path, "r") as f:
            self.config = json.load(f)

        train_config = self.config["train"]
        plot_config = self.config.get("plot", {})

        self.name = self.config["name"] # required
        self.recon_metric = train_config["recon_metric"] # required
        self.topo_coef = train_config["topo_coef"] # required
        self.optimizer = tf.keras.optimizers.Adam(train_config.get("optimizer_coef", 1e-4))

        self.save_checkpoint = train_config.get("save_checkpoint", True)
        self.checkpoint_at = train_config.get("checkpoint_at", 0)

        self.plot_sample = plot_config.get("plot_sample", False)
        self.plot_sample_size = plot_config.get("plot_sample_size", (4, 4))
        self.show_sample_at = plot_config.get("show_sample_at", 1)
        self.num_samples = self.plot_sample_size[0] * self.plot_sample_size[1]

        self.plot_latent = plot_config.get("plot_latent", False)
        self.show_latent_at = plot_config.get("show_latent_at", 1)

        self.kmprocess = kmprocess
        self._set_kmprocess_params()

        self.batch_size = 512
        self.epoch = 0

        self.history = {
            "epochs": [], 'train_loss': [],
            'elbo_loss': [], 'topo_loss': [],
            'epoch_time': []
        }

        # Train variables
        self.batch_clusters_inds = None
        self.batch_components_inds = None

    def _set_kmprocess_params(self, kmprocess=None):
        self.kmprocess = kmprocess or self.kmprocess
        if self.kmprocess is not None:
            self.num_components = self.kmprocess.num_components
            self.data2cluster_idx = self.kmprocess.data2cluster_idx
            self.cluster2component_idx = self.kmprocess.cluster2component_idx
            self.cluster_component2matrix_idx = self.kmprocess.cluster_component2matrix_idx
            self.dist_matrices = self.kmprocess.dist_matrices
            self.components_sigmas = self.kmprocess.components_sigmas
        else:
            self.num_components = None
            self.data2cluster_idx = None
            self.cluster2component_idx = None
            self.cluster_component2matrix_idx = None
            self.dist_matrices = None
            self.components_sigmas = None

    def _check_kmprocess_params(self, check_kmprocess: bool = True):
        if self.num_components is None or \
                self.data2cluster_idx is None or \
                self.cluster2component_idx is None or \
                self.cluster_component2matrix_idx is None or \
                self.dist_matrices is None or self.components_sigmas is None:
            if check_kmprocess and self.kmprocess is not None:
                self._set_kmprocess_params()
                return self._check_kmprocess_params(check_kmprocess = False)
            return False
        return True

    def _set_batch_clusters(self, batch_num, batch_len):
        if self.data2cluster_idx is None:
            self.batch_clusters_inds = np.zeros(batch_num)
            return

        st = batch_num * self.batch_size
        inds = np.arange(st, st + batch_len)
        self.batch_clusters_inds = self.data2cluster_idx[inds]

    def _set_batch_components(self, batch_num, batch_len):
        if self.cluster2component_idx is None:
            self.batch_components_inds = np.zeros(batch_num)
            return
        self._set_batch_clusters(batch_num, batch_len)
        self.batch_components_inds = self.cluster2component_idx[self.batch_clusters_inds]

    def _get_latent_dist_matrix(self, z):
        z1 = tf.reshape(z, (1, z.shape[0], z.shape[1]))
        z2 = tf.reshape(z, (z.shape[0], 1, z.shape[1]))
        dists = tf.norm(z1 - z2, ord='euclidean', axis=2)
        return dists

    def get_topo_loss(self, z, batch_num):
        total_mse = 0.0
        # Inside components regularization
        for comp_ind in range(self.num_components):
            cur_copm_selecter = self.batch_components_inds == comp_ind
            cur_comp_clusters_inds = self.batch_clusters_inds[cur_copm_selecter]
            cur_comp_z = z[cur_copm_selecter]
            if len(cur_comp_clusters_inds) > 1:
                matrix_inds = self.cluster_component2matrix_idx[cur_comp_clusters_inds]
                orig_dists = self.dist_matrices[comp_ind][matrix_inds][:, matrix_inds]

                dists = self._get_latent_dist_matrix(cur_comp_z)

                self_selector = cur_comp_clusters_inds[None, :] == cur_comp_clusters_inds[:, None]
                orig_dists[self_selector] = dists[self_selector].numpy()

                comp_mse = tf.reduce_sum(tf.square(dists - orig_dists), axis=1)
                total_mse += tf.reduce_mean(comp_mse)

        return total_mse

    def get_elbo_loss(self, x, x_recon, mean, logvar):
        if self.recon_metric == "MSE":
            recon_loss = tf.reduce_sum(tf.square(x_recon - x))
            recon_loss = recon_loss / x.shape[0]
        elif self.recon_metric == "SigmoidBinCross":
            cross_ent = tf.nn.sigmoid_cross_entropy_with_logits(logits=x_recon, labels=x)
            recon_loss = tf.reduce_sum(cross_ent, axis=[1, 2, 3])
            recon_loss= tf.reduce_mean(recon_loss)
        else:
            raise Exception(f"Not supported metric: {self.recon_metric}")

        if self.topo_coef > 0:
            KLD = 0.0
            for copm_ind in range(self.num_components):
                comp_selecter = self.batch_components_inds == copm_ind
                cur_mean, cur_logvar = mean[comp_selecter], logvar[comp_selecter]
                sigma = self.components_sigmas[copm_ind]

                KLD += 0.5 * tf.reduce_sum(
                    tf.square(cur_mean)/sigma + tf.exp(cur_logvar)/sigma - 1 - cur_logvar + tf.math.log(sigma)
                )
            KLD = KLD / x.shape[0]
        else:
            KLD = tf.reduce_mean(
                0.5 * tf.reduce_sum(tf.square(mean) + tf.exp(logvar) - 1 - logvar, axis=1)
            )

        return tf.reduce_mean(recon_loss) + KLD

    def compute_loss(self, x, batch_num):
        if self.topo_coef > 0:
            self._set_batch_components(batch_num, x.shape[0])

        if self.model.is_conditional:
            mean, logvar = self.model.encode(x, self.batch_components_inds)
            z = self.model.reparameterize(mean, logvar)
            x_recon = self.model.decode(z, self.batch_components_inds)
        else:
            mean, logvar = self.model.encode(x)
            z = self.model.reparameterize(mean, logvar)
            x_recon = self.model.decode(z)

        elbo_loss = self.get_elbo_loss(x, x_recon, mean, logvar)

        if self.topo_coef > 0:
            topo_loss = self.get_topo_loss(z, batch_num)
        else:
            topo_loss = 0.0

        total_loss = elbo_loss + self.topo_coef * topo_loss
        return total_loss, elbo_loss, topo_loss

    def train_step(self, x, batch_num):
        """Executes one training step and returns the loss.

        This function computes the loss and gradients, and uses the latter to
        update the model's parameters.
        """
        with tf.GradientTape() as tape:
            loss, elbo_loss, topo_loss = self.compute_loss(x, batch_num)

        gradients = tape.gradient(loss, self.model.trainable_variables)
        self.optimizer.apply_gradients(zip(gradients, self.model.trainable_variables))

        return loss, elbo_loss, topo_loss

    def update_history(self, train_loss, elbo_loss, topo_loss, epoch_time):
        self.history['epochs'].append(self.epoch)
        self.history['train_loss'].append(train_loss)
        self.history['elbo_loss'].append(elbo_loss)
        self.history['topo_loss'].append(topo_loss)
        self.history['epoch_time'].append(epoch_time)

    def plot_latent_space(self, dataset, colors=None):
        if self.model.is_conditional:
            dataset_components = self.cluster2component_idx[self.data2cluster_idx]
            mean, logvar = self.model.encode(dataset, dataset_components)
            z = self.model.reparameterize(mean, logvar)

            num_classes = self.num_components
            num_rows = round(np.sqrt(num_classes))
            num_cols = int(np.ceil(num_classes / num_rows))

            plt.figure(figsize=(num_cols * 4, num_rows * 4))

            for i in range(num_classes):
                plt.subplot(num_rows, num_cols, i + 1)
                cur_z = z[dataset_components == i]
                if colors is not None:
                    cur_colors = colors[dataset_components == i]
                else:
                    cur_colors = None
                plt.title(f"comp {i}")
                plt.scatter(cur_z[:, 0], cur_z[:, 1], s=0.5, c=cur_colors)
            plt.tight_layout()
        else:
            mean, logvar = self.model.encode(dataset)
            z = self.model.reparameterize(mean, logvar)

            plt.figure(figsize=(8, 8))
            plt.scatter(z[:, 0], z[:, 1], s=0.5, c=colors)

        plt.savefig(f'{self.name}_latent_at_epoch_{self.epoch:04d}.png')
        if self.epoch % self.show_latent_at == 0:
            plt.show()
        else:
            plt.close()

    def plot_sample_images(self, test_sample, test_sample_components=None):
        plt.figure(figsize=self.plot_sample_size)
        plt.rc('axes', titlesize=8)
        for i in range(test_sample.shape[0]):
            plt.subplot(*self.plot_sample_size, i + 1)
            plt.imshow(test_sample[i, :, :, 0], cmap='gray')
            if test_sample_components is not None:
                plt.title(f"comp {test_sample_components[i]}")
            plt.axis('off')
        plt.tight_layout()

    def plot_sample_images_repr(self, test_sample, test_sample_components=None):
        if self.model.is_conditional:
            mean, logvar = self.model.encode(test_sample, test_sample_components)
            z = self.model.reparameterize(mean, logvar)
            predictions = self.model.sample(test_sample_components, z)
        else:
            mean, logvar = self.model.encode(test_sample)
            z = self.model.reparameterize(mean, logvar)
            predictions = self.model.sample(z)

        self.plot_sample_images(predictions)
        plt.savefig(f'{self.name}_image_at_epoch_{self.epoch:04d}.png')
        if self.epoch % self.show_sample_at == 0:
            plt.show()
        else:
            plt.close()

    def plot_history(self):
        no_topo = self.topo_coef <= 0
        fig = plt.figure(figsize=(6, 8) if no_topo else (10, 6))
        labels = ['Train Loss', 'ELBO Loss', 'Topo Loss', 'Epoch time']
        subplot_params = (3, 1) if no_topo else (2, 2)

        for i, key in enumerate(['train_loss', 'elbo_loss', 'topo_loss', 'epoch_time']):
            if no_topo and key == 'topo_loss':
                continue
            plt.subplot(*subplot_params, (min(i, 2) if no_topo else i) + 1)
            plt.plot(self.history['epochs'], self.history[key])
            if len(self.history['epochs']) > 20:
                plt.xticks(np.linspace(
                    self.history['epochs'][0],
                    self.history['epochs'][-1], 15
                ).astype(np.uint32))
            else:
                plt.xticks(self.history['epochs'])
            plt.yticks(np.linspace(min(self.history[key]), max(self.history[key]), 10))
            plt.xlabel('Epoch')
            plt.ylabel(labels[i])

        plt.tight_layout()
        plt.savefig(f'{self.name}_train_losses_after_{self.epoch}_epochs.png')
        plt.show()

    def compress(self):
        !mkdir {self.name}
        if self.plot_sample:
            !mkdir {self.name}/Images
            !mv {self.name + '_image_at_epoch_*'} {self.name}/Images
        if self.plot_latent:
            !mkdir {self.name}/Latent
            !mv {self.name + '_latent_at_epoch_*'} {self.name}/Latent
        !mv {self.name}*.png {self.name}/
        !mv {self.name}*.weights.h5 {self.name}/
        !cp {self.config_path} {self.name}/
        !mv {self.name}*.json {self.name}/
        !zip -r {self.name}.zip {self.name}/ > /dev/null

    def train(self, X, colors=None, batch_size=512, epochs=20, compress=False, log_epochs: int = -1):
        if self.topo_coef > 0 and not self._check_kmprocess_params():
            self.kmprocess = KMProcess()
            self.kmprocess.pipeline(X, self.config_path, colors)
            assert self._check_kmprocess_params(), "Not all neeeded topo params set"

        self.model.build(X)
        self.batch_size = batch_size
        num_batches = int(np.ceil(X.shape[0] / batch_size))

        train_dataset = tf.data.Dataset.from_tensor_slices(
            X.astype(np.float32)
        ).batch(self.batch_size)

        test_sample = X[:self.num_samples] if self.plot_sample else None
        test_sample_components = None
        if test_sample is not None:
            if self.topo_coef > 0 and self.model.is_conditional:
                self._set_batch_components(0, self.num_samples)
                test_sample_components = self.batch_components_inds.copy()
            self.plot_sample_images(test_sample, test_sample_components)
            plt.savefig(f'{self.name}_orig_sample_images.png')
            plt.show()

        self.epoch += 1

        for epoch in range(self.epoch, self.epoch + epochs):
            self.epoch = epoch

            start_time = time.time()
            train_loss = tf.keras.metrics.Mean()
            elbo_losses = tf.keras.metrics.Mean()
            topo_losses = tf.keras.metrics.Mean()

            for batch_n, batch_x in enumerate(train_dataset):
                batch_start_t = time.time()
                total_loss, elbo_loss, topo_loss = self.train_step(batch_x, batch_n)
                train_loss(total_loss)
                elbo_losses(elbo_loss)
                topo_losses(topo_loss)
                batch_time = time.time() - batch_start_t
                if self.verbose > 0:
                    print(f'\rStep: {batch_n}/{num_batches}, Train Loss {total_loss:.4f}, ELBO Loss {elbo_loss:.4f}, Topo Loss {topo_loss:.4f}, time step: {batch_time:.2f}s', end='')

            total_loss = train_loss.result()
            elbo_loss = elbo_losses.result()
            topo_loss = topo_losses.result()
            total_time = time.time() - start_time
            self.update_history(total_loss, elbo_loss, topo_loss, total_time)

            if self.verbose > 0 or (log_epochs > 0 and self.epoch % log_epochs == 0):
                print('\r', end='')
                logger.info(f'Epoch: {epoch}, Train Loss {total_loss:.4f}, ELBO Loss {elbo_loss:.4f}, Topo Loss {topo_loss:.4f}, time epoch: {total_time:.2f}s')

            if self.plot_sample:
                self.plot_sample_images_repr(test_sample, test_sample_components)
            if self.plot_latent:
                self.plot_latent_space(X, colors)
            if self.save_checkpoint and self.checkpoint_at > 0:
                if epoch % self.checkpoint_at == 0:
                    self.model.save_weights(f"{self.name}_vae_epoch_{epoch}.weights.h5")

        self.plot_history()
        if self.save_checkpoint:
            self.model.save_weights(f"{self.name}_vae_epoch_{self.epoch}.weights.h5")
        if compress:
            self.compress()


# Metrics

In [ ]:
import pathlib
import fid_fix as fid

class VAEMetrics:
    INCEPTION_PATH = fid.check_or_download_inception(None)

    def __init__(self, orig_dataset=None, orig_files_path=None, orig_stats=None):
        if orig_dataset is None and orig_files_path is None and orig_stats is None:
            raise Exception("orig_dataset or orig_files_path or orig_stats should be given")
        self.orig_dataset = orig_dataset
        self.orig_files_path = orig_files_path
        self.orig_stats = orig_stats
        self.recon_stats = None

        self.name = ""
        self.is_topo = None
        self.metrics = {}

    def preprocess_array(self, arr):
        if arr.shape[-1] == 1:
            arr = np.concatenate([arr, arr, arr], axis=-1)
        elif arr.shape[-1] != 3:
            raise Exception("Invalid num channels in dataset")
        if arr[0].max() <= 1:
            arr *= 255
        return arr

    def get_orig_stats(self, sess):
        logger.info("Get orig stats")
        if self.orig_files_path is not None:
            path = pathlib.Path(self.orig_files_path)
            files = list(path.glob('*.jpg')) + list(path.glob('*.png'))
            self.orig_stats = fid.calculate_activation_statistics_from_files(files, sess)
        else:
            orig_dataset = self.preprocess_array(self.orig_dataset)
            self.orig_stats = fid.calculate_activation_statistics(orig_dataset, sess, batch_size=64, verbose=True)
        return self.orig_stats

    def get_fid(self, predicted_data):
        fid.create_inception_graph(str(self.INCEPTION_PATH))
        with tf.compat.v1.Session() as sess:
            sess.run(tf.compat.v1.global_variables_initializer())
            if self.orig_stats is None:
                self.get_orig_stats(sess)
            logger.info("Get predicted stats")
            predicted_data = self.preprocess_array(predicted_data)
            self.recon_stats = fid.calculate_activation_statistics(predicted_data, sess, batch_size=64, verbose=True)
            fid_value = fid.calculate_frechet_distance(*self.orig_stats, *self.recon_stats)
            self.metrics["FID"] = float(fid_value)
            return fid_value

    def get_mse(self, predicted_data):
        if self.orig_dataset is None:
            raise Exception("orig_dataset should be given")
        mse = np.square(predicted_data - self.orig_dataset)
        for i in range(len(predicted_data.shape) - 1, 0, -1):
            mse = np.sum(mse, axis=i)
        mse = np.mean(mse)
        self.metrics["MSE"] = float(mse)
        return mse

    def get_mae(self, predicted_data):
        if self.orig_dataset is None:
            raise Exception("orig_dataset should be given")
        mae = np.abs(predicted_data - self.orig_dataset)
        for i in range(len(predicted_data.shape) - 1, 0, -1):
            mae = np.sum(mae, axis=i)
        mae = np.mean(mae)
        self.metrics["MAE"] = float(mae)
        return mae

    def calc_metrics(self, predicted_data, config_path, metrics=None):
        with open(config_path, "r") as f:
            config = json.load(f)

        self.name = config["name"]
        self.is_topo = config["train"]["topo_coef"] > 0
        is_topo_str = "with topo" if self.is_topo else "no topo"

        if metrics is None:
            model_type = config["model"]["type"]
            metrics = ["MSE", "MAE"]
            if model_type == "ConvVAE":
                metrics += ["FID"]
        elif isinstance(metrics, str):
            metrics = [metrics]

        for metric in metrics:
            if metric == "MSE":
                metric_val = self.get_mse(predicted_data)
            elif metric == "MAE":
                metric_val = self.get_mae(predicted_data)
            elif metric == "FID":
                logger.info(f"Process {metric}...")
                metric_val = self.get_fid(predicted_data)
            else:
                logger.info("Metric not found!")
                continue
            logger.info(f"[{self.name} {is_topo_str}] {metric} = {round(float(metric_val), 4)}")

        metrics_path = f"{self.name}_metrics.json"
        metrics_dict = {"name": self.name, "is_topo": self.is_topo, "metrics": self.metrics}
        with open(metrics_path, "w") as f:
            json.dump(metrics_dict, f)
            logger.info(f"Saved metrics to {metrics_path}")

        return self.metrics

# Train Batch models

In [ ]:
def _train_model(
    data, config_path, colors, kmprocess, metrics, batch_size, epochs, model_ind
):
    model = get_model_from_config(config_path, kmprocess=kmprocess)
    pipeline = TrainPipeline(model, config_path, kmprocess=kmprocess, verbose=0)

    orig_name = pipeline.name
    pipeline.name += f"_{model_ind}"

    pipeline.train(data, colors=colors, batch_size=batch_size, epochs=epochs, compress=False, log_epochs=epochs//4)

    train_history = {
        key: [float(v) for v in val]
        for key, val in pipeline.history.items()
    }
    with open(f"{pipeline.name}_history.json", "w") as f:
        json.dump(train_history, f)
    logger.info("Train success, calc metrics...")

    if model.is_conditional:
        dataset_components = pipeline.cluster2component_idx[pipeline.data2cluster_idx]
        mean, logvar = model.encode(data, dataset_components)
    else:
        mean, logvar = model.encode(data)
    z = model.reparameterize(mean, logvar)
    X_recon = []
    for i in range(z.shape[0] // batch_size + 1):
        if model.is_conditional:
            cur_components = dataset_components[i*batch_size:(i+1)*batch_size]
            X_recon.append(
                model.sample(cur_components, z[i*batch_size:(i+1)*batch_size])
            )
        else:
            X_recon.append(model.sample(z[i*batch_size:(i+1)*batch_size]))
    X_recon = np.concatenate(X_recon, axis=0)
    metrics.calc_metrics(X_recon, config_path)
    !mv {orig_name}_metrics.json {pipeline.name}_metrics.json
    pipeline.compress()


def train_batch_models(
        data, config_path, colors=None, kmprocess=None, metrics=None,
        num_models=10, batch_size=512, epochs=200
):
    if metrics is None:
        metrics = VAEMetrics(data)

    with open(config_path, "r") as f:
        orig_name = json.load(f)["name"]

    zip_names = []
    for model_ind in range(num_models):
        logger.info(f"Process model {model_ind + 1}/{num_models}")

        _train_model(data, config_path, colors, kmprocess, metrics, batch_size, epochs, model_ind)

        zip_names.append(f"{orig_name}_{model_ind}.zip")
        logger.info(f"Done model {model_ind + 1}/{num_models}")

    !mkdir -p {orig_name}
    for zip_name in zip_names:
        !mv {zip_name} {orig_name}/
    logger.info(f"Done all models archives saved to {orig_name}/")


def train_batch_topo_and_classic_models(
    data, config_path, colors=None, kmprocess=None, metrics=None,
    num_topo_models=10, num_classic_models=10, batch_size=512, epochs=200
):
    with open(config_path, "r") as f:
        config = json.load(f)
    # Topo model
    logger.info("Train topo models")
    train_batch_models(
        data=data, config_path=config_path, colors=colors,
        kmprocess=kmprocess, metrics=metrics, num_models=num_topo_models,
        batch_size=batch_size, epochs=epochs
    )
    # Classic models
    logger.info("Train classic models")
    no_topo_config = config.copy()
    no_topo_config["name"] += "_no_topo"
    no_topo_config["train"] = config["train"].copy()
    no_topo_config["train"]["topo_coef"] = 0
    no_topo_config["model"] = config["model"].copy()
    no_topo_config["model"]["is_conditional"] = False
    no_topo_config_path = f'{no_topo_config["name"]}_config.json'
    with open(no_topo_config_path, "w") as f:
        json.dump(no_topo_config, f)

    train_batch_models(
        data=data, config_path=no_topo_config_path, colors=colors,
        kmprocess=None, metrics=metrics, num_models=num_classic_models,
        batch_size=batch_size, epochs=epochs
    )

# Config Example

In [ ]:
config = {
    # Dataset name or name for model, required
    "name": "Test",
    # Config for KMProcess
    "km": {
        # Projection method and it's params
        "project_method": "Isomap", # Isomap, TSNE, UMAP, PCA
        "project_params": {
            "n_neighbors": 10
        },
        # Cluster on original data (prefered), or on projections
        "map_use_data": True,
        # Mapping params for cover, clustering and nerve
        "map_params": {
            # Clustering method
            "cluster_method": 'DBSCAN', # DBSCAN, HDBSCAN, OPTICS
            # eps and min_samples for DBSCAN
            "eps": 1.5,
            "min_samples": 10,
            # params for cover
            "cover_n": 30,
            "cover_p": 0.4,
            # params for nerve
            "min_intersection": 1
        },
        # Calulate edges weights by data or just set them to 1
        "dists_by_data": True,
        # Scale param for each dist matrix
        # If max in dist matrix will be less then that param, it'll not change
        # If dist_matrix_scale <= 0 scale step will be skiped
        "dist_matrix_scale": 8
    },
    # Config for Model
    "model": {
        # Use CVAE (is True) or VAE (is False)
        "is_conditional": False,
        "type": "VAE", # "VAE" or "ConvVAE"
        "input_size": 3, # int or tuple
        "latent_dim": 2, # should be int
        # For VAE required, for CVAE optional and applied after flatten layer
        "hidden_layers": [16, 16],
        # Param for CVAE (required), filters in Conv2D layers, for VAE ignored
        "hidden_filters": [32, 64],
        # param for CVAE output, for VAE ignored
        "apply_sigmoid": False
        # If apply_sigmoid is True, set "recon_metric" to "MSE"
    },
    # Config for Train Pipeline
    "train": {
        # recon_metric is required and can be "MSE" or "SigmoidBinCross"
        # If used Sigmoid Binary Crossentropy, model.apply_sigmoid should be False
        "recon_metric": "MSE",
        # Topological loss coefficient, if less or equal zero topo loss won't calcualte
        "topo_coef": 0.2,
        # Learining rate for optimizer
        "optimizer_coef": 5e-3,
        # Save checkpoint (model weights) at the end of training + every `checkpoint_at` if set
        "save_checkpoint": True,
        # Save checkpoint every `checkpoint_at` epochs
        # If checkpoint_at <= 0 checkpoint will be saved only at the end of training
        "checkpoint_at": 10
    },
    # Config for plotting
    "plot": {
        # For KMProcess
        # Plot projections with params
        "plot_project": True,
        "plot_project_params": {
            "dot_size": 0.5, # deafult
            "with_legend": False
        },
        # Plot cluster graph
        "plot_graph": True,
        # For TrainPipeline
        # Plot sample images
        "plot_sample": False,
        # plot_sample is false, so next 2 params ignored
        # Grid size of sample images (number samples)
        # tuple with two params: num rows and num cols
        "plot_sample_size": (4, 4),
        # Samples images will be saved every epoch, but showed only every `show_sample_at` epochs
        # If show_sample_at <= 0, samples images won't be showed
        "show_sample_at": 0,
        # Plot latent sapce of entire dataset
        "plot_latent": True,
        # Images of latent sapce will be saved every epoch, but showed only every `show_latent_at` epochs
        # If show_latent_at <= 0, images of latent space won't be showed
        "show_latent_at": 10
    }
}

config_path = f"{config['name']}_config.json"

with open(config_path, "w") as f:
    json.dump(config, f)